# Exploratory Data Analysis: Understanding the Experiment

Before running any statistical test, we need to verify the experimental setup and understand
the data landscape. EDA answers four essential questions:

1. **Is the randomization valid?** — Sample Ratio Mismatch (SRM) check
2. **What are the headline conversion rates?** — Aggregate effect estimation
3. **Are effects homogeneous across segments?** — Heterogeneity exploration
4. **Are there temporal or distributional artifacts?** — Novelty effects, revenue outliers

Skipping EDA and jumping straight to a p-value is a common analyst mistake. A statistically
significant result in a broken experiment is meaningless. EDA is the first line of defense.

In [1]:
import matplotlib
matplotlib.use('Agg')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

df = pd.read_parquet('../data/processed/ab_data_enriched.parquet')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['date'] = df['timestamp'].dt.date
print(f"Loaded {len(df):,} rows | {df['group'].value_counts().to_dict()}")

Loaded 290,584 rows | {'treatment': 145310, 'control': 145274}


## 1. Group Balance Check (Pre-requisite for Valid Inference)

The foundational assumption of A/B testing is **random assignment**: each user has an equal
probability of being assigned to either group. If the actual split deviates significantly from
50/50, it suggests a bug in the assignment mechanism — known as **Sample Ratio Mismatch (SRM)**.

**Why SRM is serious:**
- SRM usually means the assignment is correlated with some user characteristic (e.g., browser,
  geography, time of day), which confounds the treatment effect estimate
- Even a 1-2% imbalance can bias results if the unequal groups differ systematically
- An experiment with SRM should be **invalidated and re-run**, not analyzed at face value

**Test:** Chi-squared goodness-of-fit against the expected 50/50 split. We use α=0.01
(stricter than the analysis threshold) because the cost of a false SRM alert is low,
but the cost of missing a real one is high.

In [2]:
n_control = (df['group'] == 'control').sum()
n_treatment = (df['group'] == 'treatment').sum()
total = n_control + n_treatment
expected = total / 2

chi2_srm = (n_control - expected)**2 / expected + (n_treatment - expected)**2 / expected
p_srm = 1 - stats.chi2.cdf(chi2_srm, df=1)

print(f"Control:   {n_control:,} ({n_control/total:.2%})")
print(f"Treatment: {n_treatment:,} ({n_treatment/total:.2%})")
print(f"Expected:  {expected:,.0f} each ({50:.2%})")
print(f"\nChi-squared SRM test: chi2={chi2_srm:.5f}, p={p_srm:.4f}")
print(f"Conclusion: {'BALANCED (no SRM detected)' if p_srm > 0.01 else 'WARNING: Sample Ratio Mismatch detected'}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Control', 'Treatment'], [n_control, n_treatment],
       color=['steelblue', 'coral'], alpha=0.8)
ax.axhline(expected, color='red', linestyle='--', label=f'Expected (50/50): {expected:,.0f}')
ax.set_ylabel('Users')
ax.set_title('Group Size Balance Check')
ax.legend()
for i, v in enumerate([n_control, n_treatment]):
    ax.text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

Control:   145,274 (49.99%)
Treatment: 145,310 (50.01%)
Expected:  145,292 each (5000.00%)

Chi-squared SRM test: chi2=0.00446, p=0.9468
Conclusion: BALANCED (no SRM detected)


## 2. Headline Conversion Rates

The primary metric is binary conversion (0/1). We compute the raw proportions first —
without any statistical machinery — to establish the baseline numbers that all downstream
tests will reference.

In [3]:
conv_control = df[df['group'] == 'control']['converted'].mean()
conv_treatment = df[df['group'] == 'treatment']['converted'].mean()
lift = (conv_treatment - conv_control) / conv_control * 100

print(f"Control conversion rate:    {conv_control:.4%}")
print(f"Treatment conversion rate:  {conv_treatment:.4%}")
print(f"Absolute difference:        {conv_treatment - conv_control:.4%}")
print(f"Relative lift:              {lift:.3f}%")

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    ['Control', 'Treatment'],
    [conv_control * 100, conv_treatment * 100],
    color=['steelblue', 'coral'], alpha=0.85, width=0.5
)
ax.set_ylabel('Conversion Rate (%)')
ax.set_title('Conversion Rate by Group')
ax.set_ylim(0, 15)
for bar, val in zip(bars, [conv_control, conv_treatment]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.1,
        f'{val:.2%}', ha='center', fontweight='bold'
    )
plt.tight_layout()
plt.show()

Control conversion rate:    12.0386%
Treatment conversion rate:  12.3274%
Absolute difference:        0.2888%
Relative lift:              2.399%


## 3. Conversion Rate by Segment

The aggregate +2.4% lift tells us that *on average* Treatment outperforms Control. But
averages can hide everything. A responsible analyst always decomposes aggregate effects
by relevant dimensions before making a recommendation.

**Why this matters:**
- If the lift is concentrated in one segment, a universal rollout may help that segment
  while harming others (net zero or negative)
- Segment-specific effects inform **targeting decisions** — ship to the segments where
  the new design works, hold back from segments where it does not
- This is the difference between a "ship it" and a "ship it *to these users*" recommendation

We examine four dimensions: device type, country, user segment (new vs returning), and
traffic source.

In [4]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
dimensions = ['device_type', 'country', 'user_segment', 'traffic_source']

for ax, dim in zip(axes.flatten(), dimensions):
    segment_data = []
    for val in sorted(df[dim].dropna().unique()):
        seg = df[df[dim] == val]
        ctrl_rate = seg[seg['group'] == 'control']['converted'].mean()
        treat_rate = seg[seg['group'] == 'treatment']['converted'].mean()
        segment_data.append({
            'segment': str(val),
            'control': ctrl_rate * 100,
            'treatment': treat_rate * 100
        })

    seg_df = pd.DataFrame(segment_data)
    x = np.arange(len(seg_df))
    width = 0.35
    ax.bar(x - width/2, seg_df['control'], width, label='Control', color='steelblue', alpha=0.8)
    ax.bar(x + width/2, seg_df['treatment'], width, label='Treatment', color='coral', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(seg_df['segment'], rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Conversion Rate (%)')
    ax.set_title(f'Conversion by {dim.replace("_", " ").title()}')
    ax.legend(fontsize=9)

plt.suptitle('Treatment Effect Across Segments', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. Simpson's Paradox Observation

**Simpson's Paradox** occurs when a trend that appears in aggregate data disappears —
or reverses — when the data is split by a confounding variable.

In this experiment, the aggregate shows Treatment > Control (+2.4% relative lift).
But when we split by `user_segment`, the picture changes:

- **New users** show a strong positive treatment effect: the redesigned page works well
  for first-time visitors who have no prior expectations
- **Returning users** show a neutral or slightly negative effect: they are familiar with
  the old layout, and the redesign may cause navigation friction

**Why this is Simpson's Paradox:**
The confound is that new users are more likely to convert in general (higher baseline rate)
AND respond better to Treatment. Their larger contribution to the Treatment arm (if unevenly
distributed due to timing effects) pulls the aggregate Treatment rate upward.

**Practical implication:** A universal rollout would help new users but potentially harm
retention-stage engagement with returning users. The recommendation must be segmented.

In [5]:
print("=== Simpson's Paradox: user_segment breakdown ===\n")
for seg in sorted(df['user_segment'].unique()):
    sub = df[df['user_segment'] == seg]
    ctrl = sub[sub['group'] == 'control']['converted'].mean()
    treat = sub[sub['group'] == 'treatment']['converted'].mean()
    lift_seg = (treat - ctrl) / ctrl * 100
    print(f"{seg:15s} | Control: {ctrl:.4%} | Treatment: {treat:.4%} | Lift: {lift_seg:+.2f}%")

print(f"\nAggregate:      | Control: {conv_control:.4%} | Treatment: {conv_treatment:.4%} | Lift: +{(conv_treatment - conv_control) / conv_control * 100:.2f}%")

=== Simpson's Paradox: user_segment breakdown ===

new             | Control: 11.9368% | Treatment: 12.5449% | Lift: +5.09%
returning       | Control: 12.2782% | Treatment: 11.8205% | Lift: -3.73%

Aggregate:      | Control: 12.0386% | Treatment: 12.3274% | Lift: +2.40%


## 5. Revenue and Session Duration Distributions

**Revenue** is the secondary metric. While conversion rate measures whether a user converted,
revenue measures the *magnitude* of each conversion. A design that converts fewer users at
higher order values can be more profitable than one with higher conversion and lower AOV.

**Session duration** is a guardrail metric. If the new design significantly reduces time
on page, it may indicate that users are confused or bouncing faster (negative signal), even
if conversion temporarily ticks up.

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, label, unit in [
    (axes[0], 'revenue', 'Revenue per Converter', '$'),
    (axes[1], 'session_duration_sec', 'Session Duration', 's')
]:
    for group, color in [('control', 'steelblue'), ('treatment', 'coral')]:
        data = df[df['group'] == group][metric]
        if metric == 'revenue':
            data = data[data > 0]  # Only converters have positive revenue
        ax.hist(data, bins=50, alpha=0.5, color=color, label=group.capitalize(), density=True)
        ax.axvline(data.mean(), color=color, linestyle='--', linewidth=2,
                   label=f'{group.capitalize()} mean: {data.mean():.2f} {unit}')
    ax.set_title(f'{label} Distribution')
    ax.set_xlabel(f'{label} ({unit})')
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

for metric in ['revenue', 'session_duration_sec']:
    ctrl_mean = df[df['group'] == 'control'][metric].mean()
    treat_mean = df[df['group'] == 'treatment'][metric].mean()
    print(f"{metric}: Control={ctrl_mean:.4f} | Treatment={treat_mean:.4f} | Diff={treat_mean - ctrl_mean:+.4f}")

revenue: Control=7.3765 | Treatment=8.2765 | Diff=+0.9000
session_duration_sec: Control=100.7153 | Treatment=101.1581 | Diff=+0.4428


## 6. Daily Conversion Trends

An experiment should show **stable daily conversion rates** throughout its run. Deviations
can indicate:

- **Novelty effect**: An initial spike in Treatment as curious users explore the new design,
  followed by decay. If the spike drives the aggregate significance, the long-run effect
  may be near zero.
- **Primacy effect**: The opposite — early users resist change, then adapt. The true benefit
  appears only later in the run.
- **Day-of-week effects**: Conversion rates often differ between weekdays and weekends; if
  the experiment starts mid-week, the first few days may not be representative.

Stable, overlapping trends across the 22-day window indicate that temporal confounders
are not a major concern and the experiment ran long enough to average out weekly cycles.

In [7]:
daily = df.groupby(['date', 'group']).agg(
    n=('converted', 'count'),
    conversions=('converted', 'sum')
).reset_index()
daily['conv_rate'] = daily['conversions'] / daily['n']

fig, ax = plt.subplots(figsize=(12, 5))
for group, color in [('control', 'steelblue'), ('treatment', 'coral')]:
    data = daily[daily['group'] == group].sort_values('date')
    ax.plot(
        data['date'].astype(str), data['conv_rate'] * 100,
        marker='o', markersize=4, color=color,
        label=group.capitalize(), linewidth=2
    )

ax.set_title('Daily Conversion Rate by Group')
ax.set_xlabel('Date')
ax.set_ylabel('Conversion Rate (%)')
ax.tick_params(axis='x', rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

## EDA Summary and Key Observations

The exploratory analysis reveals the following, which directly motivate the choices made in
notebook 03 (Statistical Analysis):

1. **Randomization is valid** — SRM test passes (chi2=0.00446, p=0.9468). Groups are
   balanced at ~145K each. We can proceed with causal inference.

2. **Small aggregate lift** — Treatment shows +2.4% relative lift (12.33% vs 12.04%).
   This is statistically detectable at our sample size, but Cohen's h = 0.009 places it
   firmly in the "negligible practical effect" range for conversion rate alone.

3. **Heterogeneous effects** — Mobile users and new users respond strongly to Treatment.
   Desktop users and returning users show minimal or negative effects. This heterogeneity
   argues against a universal rollout.

4. **Simpson's Paradox present** — The aggregate lift is driven primarily by new users.
   Returning users show no improvement or a slight decline. Ignoring this would lead to
   the wrong recommendation.

5. **Revenue signal is stronger than conversion rate** — Revenue per user is +$0.90
   higher in Treatment (+12.2%). This is economically meaningful even if the conversion
   lift alone might not justify the rollout cost.

6. **Daily rates are stable** — No novelty spike visible; temporal confounders appear
   minimal. The 22-day run covers approximately 3 full weekly cycles, sufficient for
   day-of-week effects to average out.